# Pauli Algebra 04: Jordan-Wigner Fermions

This notebook maps fermionic operators to symbolic Pauli sums using the Jordan-Wigner helpers.


## What you will learn

1. How spinless creation, annihilation, number, and identity operators are mapped.
2. How to verify canonical anticommutation relations in the Pauli backend.
3. How ordered fermion monomials become `PauliSum` objects.
4. How spinful site/spin labels map to Jordan-Wigner modes.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_pauli_algebra():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation.pauli_algebra")


try:
    pa = _import_pauli_algebra()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    pa = _import_pauli_algebra()

print("Loaded quantum_simulation.pauli_algebra from", Path(pa.__file__).parent)


Loaded quantum_simulation.pauli_algebra from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation/pauli_algebra


In [2]:
import numpy as np

from quantum_simulation.pauli_algebra import (
    PauliString,
    PauliSum,
    PauliHamiltonian,
    SparseKet,
    jw_spinless_fermion_operator,
    jw_spinless_fermion_product,
    jw_spinful_fermion_operator,
    jw_spinful_fermion_product,
    jw_spinful_mode_index,
    multiply_pauli_operators,
)

np.set_printoptions(precision=4, suppress=True)


## Single spinless fermion operators

The current backend uses `convention="down=1"`. Mode 0 has no Jordan-Wigner string; higher modes accumulate `Z` strings on all lower modes.


In [19]:
num_modes = 3
c0 = jw_spinless_fermion_operator("c", mode=0, num_modes=num_modes)
cdag0 = jw_spinless_fermion_operator("c_dag", mode=0, num_modes=num_modes)
c2 = jw_spinless_fermion_operator("c", mode=2, num_modes=num_modes)
n1 = jw_spinless_fermion_operator("n", mode=1, num_modes=num_modes)

print("c_0      =", c0)
print("c^dag_0  =", cdag0)
print("c_2      =", c2)


c_0      = ((0.5+0j))*XII + (0.5j)*YII
c^dag_0  = ((0.5+0j))*XII + ((-0-0.5j))*YII
c_2      = ((0.5+0j))*ZZX + (0.5j)*ZZY


In [20]:
print("n_1      =", n1)

n_1      = ((0.5+0j))*III + ((-0.5+0j))*IZI


## Canonical anticommutation checks

Because `@` means commutator, use `multiply_pauli_operators` for products when checking fermion anticommutators.


In [21]:
identity = PauliSum([PauliString("III")])
anticomm_same = (
    multiply_pauli_operators(c0, cdag0)
    + multiply_pauli_operators(cdag0, c0)
).simplify()
anticomm_diff = (
    multiply_pauli_operators(c0, c2)
    + multiply_pauli_operators(c2, c0)
).simplify()

print("{c_0, c^dag_0} =", anticomm_same)
print("{c_0, c_2}     =", anticomm_diff)
print("residual norm for identity relation:", (anticomm_same - identity).frobenius_norm())

assert (anticomm_same - identity).frobenius_norm() < 1e-10
assert len(anticomm_diff.terms) == 0


{c_0, c^dag_0} = III
{c_0, c_2}     = 0
residual norm for identity relation: 0.0


## Ordered fermion products

`jw_spinless_fermion_product` maps an ordered monomial, such as `c_i^dagger c_j`, into a single simplified `PauliSum`.


In [22]:
hop_01 = jw_spinless_fermion_product(
    operators=["c_dag", "c"],
    sites=[0, 1],
    num_modes=num_modes,
)
hop_10 = jw_spinless_fermion_product(
    operators=["c_dag", "c"],
    sites=[1, 0],
    num_modes=num_modes,
)
hermitian_hop = (hop_01 + hop_10).simplify()

print("c^dag_0 c_1 =", hop_01)
print("c^dag_0 c_1 + h.c. =", hermitian_hop)
print("Hermitian matrix:", np.allclose(hermitian_hop.to_matrix(), hermitian_hop.to_matrix().conj().T))

assert np.allclose(hermitian_hop.to_matrix(), hermitian_hop.to_matrix().conj().T)


c^dag_0 c_1 = (-0.25j)*YXI + ((0.25+0j))*YYI + ((0.25+0j))*XXI + (0.25j)*XYI
c^dag_0 c_1 + h.c. = ((0.5+0j))*YYI + ((0.5+0j))*XXI
Hermitian matrix: True


## Number operators and bit convention

Under the current convention, `n_j = (I - Z_j) / 2`. A basis state whose bit `j` is one has occupation one at mode `j`.


In [23]:
state = SparseKet.basis(num_modes, 0b010)
n0 = jw_spinless_fermion_operator("n", mode=0, num_modes=num_modes)
n1 = jw_spinless_fermion_operator("n", mode=1, num_modes=num_modes)
n2 = jw_spinless_fermion_operator("n", mode=2, num_modes=num_modes)

print("state index 0b010")
print("<n_0> =", state.measure_pauli(n0))
print("<n_1> =", state.measure_pauli(n1))
print("<n_2> =", state.measure_pauli(n2))

assert np.isclose(state.measure_pauli(n0), 0.0)
assert np.isclose(state.measure_pauli(n1), 1.0)
assert np.isclose(state.measure_pauli(n2), 0.0)


state index 0b010
<n_0> = 0j
<n_1> = (1+0j)
<n_2> = 0j


## Spinful mode mapping

A physical site has two Jordan-Wigner modes: `up -> 2 * site`, `down -> 2 * site + 1`.


In [24]:
physical_L = 3
for site in range(physical_L):
    print(
        f"site {site}: up mode {jw_spinful_mode_index(site, 'up', physical_L)}, "
        f"down mode {jw_spinful_mode_index(site, 'down', physical_L)}"
    )

n_site1_up = jw_spinful_fermion_operator("n", site=1, spin="up", physical_lattice_length=physical_L)
print("n(site=1, up) =", n_site1_up)
assert jw_spinful_mode_index(1, "up", physical_L) == 2
assert jw_spinful_mode_index(1, "down", physical_L) == 3


site 0: up mode 0, down mode 1
site 1: up mode 2, down mode 3
site 2: up mode 4, down mode 5
n(site=1, up) = ((0.5+0j))*IIIIII + ((-0.5+0j))*IIZIII


## Building a fermion Hamiltonian through `PauliHamiltonian`

The builder wraps the Jordan-Wigner product helpers and merges all resulting Pauli terms.


In [25]:
L = 3
t = 1.0
builder = PauliHamiltonian(lattice_length=L)
for j in range(L - 1):
    builder.add_fermion_term(-t, ["c_dag", "c"], [j, j + 1])
    builder.add_fermion_term(-t, ["c_dag", "c"], [j + 1, j])

H_hopping = builder.build()
H_mat = H_hopping.to_matrix()
print("spinless hopping Hamiltonian:", H_hopping)
print("term count:", builder.num_terms())
print("Hermitian:", np.allclose(H_mat, H_mat.conj().T))

assert np.allclose(H_mat, H_mat.conj().T)


spinless hopping Hamiltonian: ((-0.5+0j))*YYI + ((-0.5+0j))*XXI + ((-0.5+0j))*IYY + ((-0.5+0j))*IXX
term count: 4
Hermitian: True


## Spinful ordered products

The spinful helper accepts physical sites plus spin labels, then maps them to the underlying Jordan-Wigner mode chain.


In [26]:
spinful_hop = jw_spinful_fermion_product(
    operators=["c_dag", "c"],
    sites=[0, 1],
    spins=["up", "up"],
    physical_lattice_length=2,
)
print("c^dag_{0,up} c_{1,up} =", spinful_hop)

spinful_builder = PauliHamiltonian(lattice_length=4)
spinful_builder.add_spinful_fermion_term(
    -1.0,
    operators=["c_dag", "c"],
    sites=[0, 1],
    spins=["up", "up"],
    physical_lattice_length=2,
)
spinful_builder.add_spinful_fermion_term(
    -1.0,
    operators=["c_dag", "c"],
    sites=[1, 0],
    spins=["up", "up"],
    physical_lattice_length=2,
)
print("builder h + h.c. =", spinful_builder.build())


c^dag_{0,up} c_{1,up} = (-0.25j)*YZXI + ((0.25+0j))*YZYI + ((0.25+0j))*XZXI + (0.25j)*XZYI
builder h + h.c. = ((-0.5+0j))*YZYI + ((-0.5+0j))*XZXI
